# Investigating Negation Sensitivity in DistilBERT Sentiment Classification

## 1. Project Overview

This investigation examines how negation affects the predictions and confidence of a pretrained sentiment classification model.

We use DistilBERT fine-tuned on the Stanford Sentiment Treebank (SST-2) and investigate its behavior by comparing original sentences with controlled negated versions.

### Research Question

How does introducing negation into sentiment-bearing sentences affect the predictions and confidence of DistilBERT?

### Objectives

1. Investigate how negation affects positive and negative sentiment probabilities.
2. Compare the effects of negation on originally positive and negative sentences.
3. Identify examples where the model does not respond as expected to negation.
4. Investigate whether prediction confidence changes consistently with predicted sentiment.

### Initial Hypotheses

- H1: Adding negation to originally positive sentences will generally decrease their positive sentiment probability.
- H2: The effect of negation may differ depending on the original sentiment.
- H3: Some sentences may receive highly confident predictions that disagree with human-annotated sentiment.

### Experimental Setup

- **Task:** Binary sentiment classification.
- **Dataset:** SST-2 validation subset with manually reviewed negation variants.
- **Model:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english`
- **Training:** No additional training or fine-tuning.
- **Approach:** Black-box behavioral investigation using input-output comparisons.

## 2. Environment Setup

In [1]:
%pip install -q datasets transformers torch pandas plotly -U nbformat ipython

Note: you may need to restart the kernel to use updated packages.


## 3. Dataset Loading and Exploration

In [2]:
from datasets import load_dataset
import pandas as pd

# Load the official SST-2 dataset
dataset = load_dataset("stanfordnlp/sst2")

# Inspect the available splits
print(dataset)

# Display five validation examples
validation_df = pd.DataFrame(dataset["validation"][:5])

validation_df["label_name"] = validation_df["label"].map({
    0: "negative",
    1: "positive"
})

display(validation_df[["sentence", "label_name"]])

d:\JKU\sem 2\AI & vis\sentiment-perturbation-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


,sentence,label_name
0,it 's a charming and often affecting journey .,positive
1,unflinchingly bleak and desperate,negative
2,allows us to hope that nolan is poised to emba...,positive
3,"the acting , costumes , music , cinematography...",positive
4,"it 's slow -- very , very slow .",negative


## 4. Model Initialization

In [3]:
from transformers import pipeline

MODEL_ID = (
    "distilbert/"
    "distilbert-base-uncased-finetuned-sst-2-english"
)

classifier = pipeline(
    "text-classification",
    model=MODEL_ID,
    top_k=None
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1000.01it/s]


Model loaded successfully!


## 5. Initial Model Validation

In [4]:
test_texts = [
    "The movie was wonderful.",
    "The movie was terrible."
]

predictions = classifier(test_texts)

for text, scores in zip(test_texts, predictions):
    print(f"\nSentence: {text}")

    for result in scores:
        print(
            f"  {result['label']}: "
            f"{result['score']:.4f}"
        )


Sentence: The movie was wonderful.
  POSITIVE: 0.9999
  NEGATIVE: 0.0001

Sentence: The movie was terrible.
  NEGATIVE: 0.9998
  POSITIVE: 0.0002


## 6. Initial Negation Experiments

In [6]:
test_pairs = [
    (
        "The movie was okay.",
        "The movie was not okay."
    ),
    (
        "The story and structure are well-honed.",
        "The story and structure are not well-honed."
    ),
    (
        "The acting was excellent.",
        "The acting was not excellent."
    ),
]

for original, modified in test_pairs:
    results = classifier([original, modified])

    def get_scores(scores):
        return {
            item["label"]: item["score"]
            for item in scores
        }

    original_scores = get_scores(results[0])
    modified_scores = get_scores(results[1])

    print(f"\nOriginal: {original}")
    print(f"Modified: {modified}")

    print(
        "Original positive probability:",
        round(original_scores["POSITIVE"], 4)
    )
    print(
        "Modified positive probability:",
        round(modified_scores["POSITIVE"], 4)
    )
    print(
        "Change in positive probability:",
        round(
            modified_scores["POSITIVE"]
            - original_scores["POSITIVE"],
            4
        )
    )


Original: The movie was okay.
Modified: The movie was not okay.
Original positive probability: 0.9998
Modified positive probability: 0.0002
Change in positive probability: -0.9996



Original: The story and structure are well-honed.
Modified: The story and structure are not well-honed.
Original positive probability: 0.9998
Modified positive probability: 0.0003
Change in positive probability: -0.9995

Original: The acting was excellent.
Modified: The acting was not excellent.
Original positive probability: 0.9999
Modified positive probability: 0.0003
Change in positive probability: -0.9996


In [7]:
import pandas as pd

# Convert the validation split into a DataFrame
validation_df = pd.DataFrame(dataset["validation"])

# Add readable sentiment labels
validation_df["label_name"] = validation_df["label"].map({
    0: "negative",
    1: "positive"
})

# Calculate sentence length
validation_df["word_count"] = (
    validation_df["sentence"].str.split().str.len()
)

# Keep relatively short sentences for easier controlled edits
candidates = validation_df[
    validation_df["word_count"].between(4, 12)
]

# Display 10 positive and 10 negative candidates
for sentiment, label in [("Positive", 1), ("Negative", 0)]:
    print(f"\n{sentiment} candidates:")

    sample = (
        candidates[candidates["label"] == label]
        .sample(n=10, random_state=42)
    )

    display(sample[["sentence", "label_name", "word_count"]])


Positive candidates:


,sentence,label_name,word_count
773,it 's a beautiful madness .,positive,6
741,but it still jingles in the pocket .,positive,8
679,filmmakers who can deftly change moods are tre...,positive,12
848,his healthy sense of satire is light and fun ...,positive,10
306,the story and structure are well-honed .,positive,7
746,people cinema at its finest .,positive,6
643,"the jabs it employs are short , carefully plac...",positive,12
383,a warm but realistic meditation on friendship ...,positive,12
0,it 's a charming and often affecting journey .,positive,9
112,hilariously inept and ridiculous .,positive,5



Negative candidates:


,sentence,label_name,word_count
631,burns never really harnesses to full effect th...,negative,11
58,manages to be both repulsively sadistic and mu...,negative,9
18,... the movie is just a plain old monster .,negative,10
650,"too slow , too long and too little happens .",negative,10
539,do not see this film .,negative,6
578,just not campy enough,negative,4
256,rarely has so much money delivered so little e...,negative,10
377,nothing is sacred in this gut-buster .,negative,7
751,one long string of cliches .,negative,6
74,a string of rehashed sight gags based in insip...,negative,11


### Initial Observations

In the initial exploratory experiment, three sentence pairs were evaluated using the pretrained DistilBERT classifier.

All three modified sentences showed a substantial decrease in positive sentiment probability after negation was introduced.

These observations suggest that negation can strongly influence the model's predictions. However, the sample is too small to establish systematic behavior.

A larger and more controlled dataset is required to investigate whether this pattern is consistent across different sentences and original sentiment classes.

## 7. Experimental Dataset Construction

We construct a controlled dataset using sentences from the SST-2 validation split.

The dataset contains originally positive and negative sentences that can naturally accommodate syntactic negation.

For each selected sentence, we create a modified version by inserting "not" after the first occurrence of a suitable auxiliary verb (is, are, was, or were).

This allows us to compare the model's predictions before and after negation while keeping the original sentence structure as consistent as possible.

All the generated sentence pairs are reviewed for grammatical
correctness and semantic suitability. The review process
includes GenAI-assisted filtering and human verification.

In [8]:
import re

# Recreate validation DataFrame and preserve original indices
validation_df = pd.DataFrame(dataset["validation"])
validation_df["sst2_index"] = validation_df.index

validation_df["label_name"] = validation_df["label"].map({
    0: "negative",
    1: "positive"
})

validation_df["word_count"] = (
    validation_df["sentence"].str.split().str.len()
)

# Identify suitable auxiliary verbs
be_pattern = re.compile(
    r"\b(is|are|was|were)\b",
    flags=re.IGNORECASE
)

# Exclude sentences that already contain common negation markers
negation_pattern = re.compile(
    r"\b(not|never|no|none|nothing|neither|nor|"
    r"rarely|hardly|scarcely|without)\b|\b\w+n't\b",
    flags=re.IGNORECASE
)

# Filter suitable original sentences
eligible = validation_df[
    validation_df["word_count"].between(5, 18)
    & validation_df["sentence"].apply(
        lambda x: bool(be_pattern.search(x))
        and not bool(negation_pattern.search(x))
    )
].copy()

print("Eligible candidates by sentiment:")
print(eligible["label_name"].value_counts())

Eligible candidates by sentiment:
label_name
positive    55
negative    41
Name: count, dtype: int64


In [9]:
def introduce_negation(sentence):
    return be_pattern.sub(
        lambda match: match.group(0) + " not",
        sentence,
        count=1
    )


TARGET_PER_CLASS = 40

candidate_groups = []

for sentiment in ["positive", "negative"]:

    group = eligible[
        eligible["label_name"] == sentiment
    ]

    selected = group.sample(
        n=min(TARGET_PER_CLASS, len(group)),
        random_state=42
    ).copy()

    selected["modified_sentence"] = (
        selected["sentence"].apply(introduce_negation)
    )

    selected["source"] = "SST-2 validation"

    candidate_groups.append(selected)

# Combine positive and negative candidates
candidate_pool = pd.concat(
    candidate_groups,
    ignore_index=True
)

candidate_pool = candidate_pool.rename(columns={
    "sentence": "original_sentence",
    "label_name": "original_label"
})

candidate_pool["review_status"] = "pending"
candidate_pool["modified_human_label"] = ""

# Display the generated pairs
display(
    candidate_pool[
        [
            "sst2_index",
            "original_label",
            "original_sentence",
            "modified_sentence",
            "review_status"
        ]
    ]
)

print("Total candidate pairs:", len(candidate_pool))

,sst2_index,original_label,original_sentence,modified_sentence,review_status
0,468,positive,the characters are interesting and often very ...,the characters are not interesting and often v...,pending
1,66,positive,"if you 're hard up for raunchy college humor ,...","if you 're hard up for raunchy college humor ,...",pending
2,470,positive,reggio 's continual visual barrage is absorbin...,reggio 's continual visual barrage is not abso...,pending
3,169,positive,leigh 's film is full of memorable performance...,leigh 's film is not full of memorable perform...,pending
4,286,positive,"at its best , queen is campy fun like the vinc...","at its best , queen is not campy fun like the ...",pending
...,...,...,...,...,...
75,413,negative,at least one scene is so disgusting that viewe...,at least one scene is not so disgusting that v...,pending
76,471,negative,adults will wish the movie were less simplisti...,adults will wish the movie were not less simpl...,pending
77,153,negative,the heavy-handed film is almost laughable as a...,the heavy-handed film is not almost laughable ...,pending
78,368,negative,"the humor is forced and heavy-handed , and occ...","the humor is not forced and heavy-handed , and...",pending


Total candidate pairs: 80


Export for manual review

In [10]:
# Prepare a clean review DataFrame
review_df = candidate_pool[
    [
        "sst2_index",
        "original_label",
        "original_sentence",
        "modified_sentence",
        "source"
    ]
].copy()

# Add review status
review_df["review_status"] = "pending"

# Create data directory
from pathlib import Path

Path("data").mkdir(exist_ok=True)

# Export CSV
review_df.to_csv(
    "data/phase1_negation_review.csv",
    index=False
)

print("Review file created!")

Review file created!


Upload the reviewd file

In [11]:
reviewed_df = pd.read_csv(
    "data/phase1_negation_reviewed_ai_v1.csv"
)

final_pairs = reviewed_df[
    reviewed_df["review_status"] == "accepted"
].copy()

print("Accepted pairs:", len(final_pairs))

print(
    final_pairs["original_label"].value_counts()
)

Accepted pairs: 51
original_label
positive    29
negative    22
Name: count, dtype: int64


## 8. Model Inference

In this section, the pretrained DistilBERT classifier is applied to all accepted sentence pairs.

For each original and modified sentence, we collect the predicted sentiment and the probabilities assigned to both sentiment classes.

This allows us to investigate how introducing negation affects the model's predictions and confidence.

In [12]:
# Reset index and create pair IDs
final_pairs = final_pairs.reset_index(drop=True).copy()

final_pairs["pair_id"] = [
    f"N{i+1:03d}" for i in range(len(final_pairs))
]

# Normalize labels
final_pairs["original_label"] = (
    final_pairs["original_label"].str.lower().str.strip()
)

print("Accepted pairs:", len(final_pairs))
print("\nOriginal sentiment distribution:")
print(final_pairs["original_label"].value_counts())

display(final_pairs.head())

Accepted pairs: 51

Original sentiment distribution:
original_label
positive    29
negative    22
Name: count, dtype: int64


,sst2_index,original_label,original_sentence,modified_sentence,source,review_status,pair_id
0,468,positive,the characters are interesting and often very ...,the characters are not interesting and often v...,SST-2 validation,accepted,N001
1,66,positive,"if you 're hard up for raunchy college humor ,...","if you 're hard up for raunchy college humor ,...",SST-2 validation,accepted,N002
2,169,positive,leigh 's film is full of memorable performance...,leigh 's film is not full of memorable perform...,SST-2 validation,accepted,N003
3,286,positive,"at its best , queen is campy fun like the vinc...","at its best , queen is not campy fun like the ...",SST-2 validation,accepted,N004
4,718,positive,"this is human comedy at its most amusing , int...","this is not human comedy at its most amusing ,...",SST-2 validation,accepted,N005


In [13]:
# Collect all original and modified sentences
original_texts = final_pairs["original_sentence"].tolist()
modified_texts = final_pairs["modified_sentence"].tolist()

all_texts = original_texts + modified_texts

# Run inference
all_predictions = classifier(
    all_texts,
    batch_size=16
)

print("Total predictions:", len(all_predictions))

Total predictions: 102


Extract probabilities and calculate changes

In [14]:
def get_scores(scores):
    return {
        item["label"].upper(): float(item["score"])
        for item in scores
    }


results = []

n = len(final_pairs)

for i, row in final_pairs.iterrows():

    original_scores = get_scores(all_predictions[i])
    modified_scores = get_scores(all_predictions[i + n])

    original_prediction = max(
        original_scores,
        key=original_scores.get
    )

    modified_prediction = max(
        modified_scores,
        key=modified_scores.get
    )

    original_confidence = max(original_scores.values())
    modified_confidence = max(modified_scores.values())

    results.append({
        **row.to_dict(),

        "original_prediction": original_prediction,
        "modified_prediction": modified_prediction,

        "original_positive_probability":
            original_scores["POSITIVE"],

        "modified_positive_probability":
            modified_scores["POSITIVE"],

        "original_negative_probability":
            original_scores["NEGATIVE"],

        "modified_negative_probability":
            modified_scores["NEGATIVE"],

        "original_confidence": original_confidence,
        "modified_confidence": modified_confidence,

        "positive_probability_change":
            modified_scores["POSITIVE"]
            - original_scores["POSITIVE"],

        "confidence_change":
            modified_confidence
            - original_confidence,

        "prediction_flipped":
            original_prediction != modified_prediction,

        "original_correct":
            original_prediction == row["original_label"].upper()
    })

results_df = pd.DataFrame(results)

display(results_df.head())

,sst2_index,original_label,original_sentence,modified_sentence,source,review_status,pair_id,original_prediction,modified_prediction,original_positive_probability,modified_positive_probability,original_negative_probability,modified_negative_probability,original_confidence,modified_confidence,positive_probability_change,confidence_change,prediction_flipped,original_correct
0,468,positive,the characters are interesting and often very ...,the characters are not interesting and often v...,SST-2 validation,accepted,N001,POSITIVE,NEGATIVE,0.999802,0.001688,0.000198,0.998312,0.999802,0.998312,-0.998114,-0.001490,True,True
1,66,positive,"if you 're hard up for raunchy college humor ,...","if you 're hard up for raunchy college humor ,...",SST-2 validation,accepted,N002,POSITIVE,NEGATIVE,0.831662,0.000616,0.168338,0.999384,0.831662,0.999384,-0.831046,0.167722,True,True
2,169,positive,leigh 's film is full of memorable performance...,leigh 's film is not full of memorable perform...,SST-2 validation,accepted,N003,POSITIVE,NEGATIVE,0.999841,0.000313,0.000159,0.999687,0.999841,0.999687,-0.999528,-0.000154,True,True
3,286,positive,"at its best , queen is campy fun like the vinc...","at its best , queen is not campy fun like the ...",SST-2 validation,accepted,N004,POSITIVE,NEGATIVE,0.999830,0.016454,0.000170,0.983546,0.999830,0.983546,-0.983376,-0.016284,True,True
4,718,positive,"this is human comedy at its most amusing , int...","this is not human comedy at its most amusing ,...",SST-2 validation,accepted,N005,POSITIVE,NEGATIVE,0.999828,0.017901,0.000172,0.982099,0.999828,0.982099,-0.981927,-0.017730,True,True


In [15]:
from pathlib import Path

results_dir = Path("results/genai_pair")
results_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    results_dir / "negation_predictions.csv",
    index=False
)

print("Inference results saved!")

Inference results saved!


## 9. Behavioral Analysis

We examine how DistilBERT responds to negation by comparing its predictions and confidence for original and modified sentences.

The analysis focuses on differences between originally positive and negative sentences, prediction flips, and changes in positive sentiment probability and model confidence.

Compare the two groups

In [16]:
summary = results_df.groupby("original_label").agg(
    number_of_pairs=("pair_id", "count"),

    mean_original_positive=(
        "original_positive_probability", "mean"
    ),

    mean_modified_positive=(
        "modified_positive_probability", "mean"
    ),

    mean_probability_change=(
        "positive_probability_change", "mean"
    ),

    prediction_flip_rate=(
        "prediction_flipped", "mean"
    ),

    mean_confidence_change=(
        "confidence_change", "mean"
    )
).reset_index()

summary["prediction_flip_rate"] *= 100

display(
    summary.round(4)
)

,original_label,number_of_pairs,mean_original_positive,mean_modified_positive,mean_probability_change,prediction_flip_rate,mean_confidence_change
0,negative,22,0.0914,0.5086,0.4172,54.5455,-0.0551
1,positive,29,0.9593,0.1385,-0.8208,89.6552,-0.0182


## 10. Interactive Visualization

### 10.1 Original vs. Modified Sentiment Probability

This scatter plot compares the positive sentiment probability assigned to each original sentence against the probability assigned to its negated version.

Each point represents one sentence pair.

The visualization allows us to investigate how negation influences sentiment predictions, distinguish originally positive and negative sentences, and identify examples where the model's behavior differs from the expected pattern.

In [17]:
import plotly.express as px

fig = px.scatter(
    results_df,

    x="original_positive_probability",
    y="modified_positive_probability",

    color="original_label",
    symbol="prediction_flipped",

    hover_name="pair_id",

    hover_data={
        "original_sentence": True,
        "modified_sentence": True,
        "original_prediction": True,
        "modified_prediction": True,
        "positive_probability_change": ":.4f",
        "confidence_change": ":.4f",
        "original_label": True
    },

    title="Effect of Negation on Positive Sentiment Probability",

    labels={
        "original_positive_probability":
            "Original P(Positive)",

        "modified_positive_probability":
            "Modified P(Positive)",

        "original_label":
            "Original Sentiment",

        "prediction_flipped":
            "Prediction Changed"
    }
)

# Reference: no probability change
fig.add_shape(
    type="line",
    x0=0, y0=0,
    x1=1, y1=1,
    line=dict(color="gray", dash="dash")
)

# Binary classification thresholds
fig.add_hline(
    y=0.5,
    line_dash="dot",
    line_color="gray",
    opacity=0.5
)

fig.add_vline(
    x=0.5,
    line_dash="dot",
    line_color="gray",
    opacity=0.5
)

fig.update_layout(
    xaxis=dict(range=[-0.03, 1.03]),
    yaxis=dict(range=[-0.03, 1.03]),
    height=650,
    width=900
)

fig.show()

### 10.2 Distribution of Probability Changes

This box plot compares changes in positive sentiment probability across originally positive and negative sentences.

It helps identify differences between the two groups, examine the spread of probability changes, and identify unusual examples.

In [18]:
fig2 = px.box(
    results_df,

    x="original_label",
    y="positive_probability_change",

    color="original_label",

    points="all",

    hover_name="pair_id",

    hover_data={
        "original_sentence": True,
        "modified_sentence": True,
        "original_prediction": True,
        "modified_prediction": True,
        "positive_probability_change": ":.4f"
    },

    title="Distribution of Positive Probability Changes After Negation",

    labels={
        "original_label": "Original Sentiment",
        "positive_probability_change":
            "Change in P(Positive)"
    }
)

fig2.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

fig2.update_layout(
    height=550,
    width=850
)

fig2.show()

In [19]:
# Largest decreases in positive probability
print("Largest decreases:")

display(
    results_df.nsmallest(
        5,
        "positive_probability_change"
    )[
        [
            "original_sentence",
            "modified_sentence",
            "original_prediction",
            "modified_prediction",
            "positive_probability_change",
            "confidence_change"
        ]
    ]
)

# Largest increases in positive probability
print("Largest increases:")

display(
    results_df.nlargest(
        5,
        "positive_probability_change"
    )[
        [
            "original_sentence",
            "modified_sentence",
            "original_prediction",
            "modified_prediction",
            "positive_probability_change",
            "confidence_change"
        ]
    ]
)

Largest decreases:


,original_sentence,modified_sentence,original_prediction,modified_prediction,positive_probability_change,confidence_change
8,"the film is powerful , accessible and funny .","the film is not powerful , accessible and funn...",POSITIVE,NEGATIVE,-0.999613,-0.000162
5,"it is amusing , and that 's all it needs to be .","it is not amusing , and that 's all it needs t...",POSITIVE,NEGATIVE,-0.999590,-0.000061
10,the draw ( for `` big bad love '' ) is a solid...,the draw ( for `` big bad love '' ) is not a s...,POSITIVE,NEGATIVE,-0.999588,-0.000117
16,preaches to two completely different choirs at...,preaches to two completely different choirs at...,POSITIVE,NEGATIVE,-0.999564,-0.000129
28,the story and structure are well-honed .,the story and structure are not well-honed .,POSITIVE,NEGATIVE,-0.999539,-0.000069


Largest increases:


,original_sentence,modified_sentence,original_prediction,modified_prediction,positive_probability_change,confidence_change
29,this is so bad .,this is not so bad .,NEGATIVE,POSITIVE,0.999303,-0.000259
34,do n't be fooled by the impressive cast list -...,do n't be fooled by the impressive cast list -...,NEGATIVE,POSITIVE,0.997281,0.000677
41,this movie is maddening .,this movie is not maddening .,NEGATIVE,POSITIVE,0.996688,-0.002889
46,"every time you look , sweet home alabama is ta...","every time you look , sweet home alabama is no...",NEGATIVE,POSITIVE,0.995959,0.000607
39,whaley 's determination to immerse you in shee...,whaley 's determination to immerse you in shee...,NEGATIVE,POSITIVE,0.993666,-0.004792


In [20]:
results_df["flipped_to_positive"] = (
    (results_df["original_prediction"] == "NEGATIVE") &
    (results_df["modified_prediction"] == "POSITIVE")
)

results_df["flipped_to_negative"] = (
    (results_df["original_prediction"] == "POSITIVE") &
    (results_df["modified_prediction"] == "NEGATIVE")
)

flip_analysis = results_df.groupby("original_label").agg(
    pairs=("pair_id", "count"),

    original_accuracy=("original_correct", "mean"),

    flipped_to_positive=("flipped_to_positive", "mean"),

    flipped_to_negative=("flipped_to_negative", "mean"),

    unchanged_prediction=("prediction_flipped",
                          lambda x: (~x).mean())
).reset_index()

for col in [
    "original_accuracy",
    "flipped_to_positive",
    "flipped_to_negative",
    "unchanged_prediction"
]:
    flip_analysis[col] *= 100

display(flip_analysis.round(2))

,original_label,pairs,original_accuracy,flipped_to_positive,flipped_to_negative,unchanged_prediction
0,negative,22,90.91,50.00,4.55,45.45
1,positive,29,96.55,3.45,86.21,10.34


### 10.3 Prediction Transition Analysis

This visualization examines how the predicted sentiment changes after negation is introduced.

It compares originally positive and negative sentences and distinguishes between predictions that change from negative to positive, positive to negative, and predictions that remain unchanged.

This allows us to investigate whether the model responds differently to negation depending on the original sentiment of the sentence.

In [21]:
# Define prediction transitions
results_df["prediction_transition"] = results_df.apply(
    lambda row:
        f"{row['original_prediction']} → {row['modified_prediction']}"
        if row["prediction_flipped"]
        else "Unchanged",
    axis=1
)

# Define the transition categories
transition_order = [
    "POSITIVE → NEGATIVE",
    "NEGATIVE → POSITIVE",
    "Unchanged"
]

# Count transitions for each original sentiment
transition_counts = (
    results_df.groupby(
        ["original_label", "prediction_transition"],
        observed=False
    )
    .size()
    .reset_index(name="count")
)

# Include categories with zero observations
all_combinations = pd.MultiIndex.from_product(
    [
        ["negative", "positive"],
        transition_order
    ],
    names=["original_label", "prediction_transition"]
).to_frame(index=False)

transition_df = all_combinations.merge(
    transition_counts,
    on=["original_label", "prediction_transition"],
    how="left"
)

transition_df["count"] = (
    transition_df["count"].fillna(0).astype(int)
)

# Calculate percentages within each original sentiment group
transition_df["percentage"] = (
    transition_df["count"]
    / transition_df.groupby("original_label")["count"]
        .transform("sum")
    * 100
)

# Create visualization
fig3 = px.bar(
    transition_df,
    x="original_label",
    y="percentage",
    color="prediction_transition",
    barmode="stack",
    text="percentage",
    category_orders={
        "original_label": ["negative", "positive"],
        "prediction_transition": transition_order
    },
    title="Prediction Transitions After Negation",
    labels={
        "original_label": "Original Sentiment",
        "percentage": "Percentage of Sentence Pairs",
        "prediction_transition": "Prediction Transition"
    },
    hover_data={
        "count": True,
        "percentage": ":.2f"
    }
)

fig3.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="inside"
)

fig3.update_layout(
    height=550,
    width=900,
    yaxis=dict(
        title="Percentage of Sentence Pairs",
        range=[0, 100]
    ),
    legend_title="Prediction Transition"
)

fig3.show()

### 10.4 Confidence Changes After Negation

This visualization compares changes in predicted-class confidence between originally positive and negative sentences.

It uses box plots to examine the distribution of confidence changes and individual points to identify sentences that behave differently from their respective groups.

The analysis aims to determine whether negation affects model confidence consistently across both original sentiment classes.

In [23]:
fig4 = px.box(
    results_df,

    x="original_label",
    y="confidence_change",

    color="original_label",

    points="all",

    hover_name="pair_id",

    hover_data={
        "original_sentence": True,
        "modified_sentence": True,
        "original_prediction": True,
        "modified_prediction": True,
        "original_confidence": ":.4f",
        "modified_confidence": ":.4f",
        "confidence_change": ":.4f"
    },

    category_orders={
        "original_label": ["negative", "positive"]
    },

    title="Changes in Model Confidence After Negation",

    labels={
        "original_label": "Original Sentiment",
        "confidence_change":
            "Change in Predicted-Class Confidence"
    }
)

# Reference line: no confidence change
fig4.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

fig4.update_layout(
    height=600,
    width=900,
    showlegend=False
)

fig4.show()